# Lesson 6.1: Mapping Fundamentals

**🎬 Video:** [Lesson 6.1: Mapping Fundamentals](#)

## Overview

This is the culminating lesson of the unit. Over the last several lessons you have:

- **Lesson 3** — Loaded and cleaned raw Reddit data
- **Lesson 4** — Extracted place names using NER models and resolved them to coordinates with a geoparser
- **Lesson 5** — Scored each sentence for sentiment using VADER and RoBERTa

Now you have two things attached to each sentence: a **location** and an **emotion**. In this lesson you will aggregate those scores by place and put them on a map. This map will form the basis for later analysis. 

This lesson will show you some of the critical decisions that go into making a map. For the final project there is no "right" decision, but you will have to justify why you designed the map the way you did. It needs to be representative of the underlying data and the story you are trying to tell.



---

## 1 Load the Data

Load the sentiment dataset your team produced in Lesson 5.2. If the file is not in memory, load it from the pickle file your teammate committed to the repository.


In [ ]:
import sys
import pandas as pd
import plotly.express as px
import numpy as np

sys.path.insert(0, "..")
from tests.helpers import apply_review_corrections

# ── 1. Load sentence-level sentiment data ───────────────────────────────────
# ROBERTA_COMPLETE tracks whether this session is using the group's own
# processed data (True) or the instructor backup (False).
# Autograding checks for the team file — backup data does NOT earn credit.
ROBERTA_COMPLETE = True

print("📥 Loading from shared data folder...")
try:
    df_reddit_sentiment_full = pd.read_pickle("../data/JMU/jmu_geoparsed_cleaned_sentiment.pickle")
    print(f"✅ Loaded {len(df_reddit_sentiment_full):,} rows")
except FileNotFoundError:
    print("⚠️  Team file not found — loading instructor backup so you can continue")
    print("   Your group still needs to complete Lesson 5.2")
    print("   and commit ../data/JMU/jmu_geoparsed_cleaned_sentiment.pickle before this counts.")
    try:
        df_reddit_sentiment_full = pd.read_pickle(
            "../data/JMU/JMU_geoparsed_cleaned_sentiment_backup.pickle"
        )
        ROBERTA_COMPLETE = False
        print(f"⚠️  Backup loaded ({len(df_reddit_sentiment_full):,} rows) — RoBERTa step NOT credited")
    except FileNotFoundError:
        print("❌ Backup also not found — ask your instructor for help.")
        df_reddit_sentiment_full = pd.DataFrame(
            columns=["place", "latitude", "longitude", "sentences",
                     "roberta_compound", "place_type"]
        )
        ROBERTA_COMPLETE = False

# ── 2. Apply review corrections ──────────────────────────────────────────────
_df = apply_review_corrections(df_reddit_sentiment_full)

# ── 3. Aggregate to place-level summary ─────────────────────────────────────
df_reddit_place_sentiments = (
    _df
    .dropna(subset=["place", "latitude", "longitude"])
    .astype({"latitude": float, "longitude": float})
    .groupby("place", sort=False)
    .agg(
        location_count=("place", "size"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
        sentences=("sentences", lambda x: " | ".join(str(s) for s in list(x)[:5])),
        avg_roberta_compound=("roberta_compound", "mean"),
        reviewer=("reviewer", "first")
    )
    .reset_index()
)

# ── 4. Merge place_type ──────────────────────────────────────────────────────
if "place_type" in _df.columns:
    _ref = (
        _df.loc[_df["place_type"].fillna("").ne(""), ["place", "place_type"]]
        .drop_duplicates("place")
    )
else:
    _ref = (
        pd.read_csv("../data/JMU/JMU_geoparsed_long.csv", usecols=["place", "place_type"])
        .drop_duplicates("place")
    )
df_reddit_place_sentiments = df_reddit_place_sentiments.merge(_ref, on="place", how="left")

df_reddit_place_sentiments["reviewer"] = df_reddit_place_sentiments.pop("reviewer")
df_reddit_place_sentiments["reviewer"] = df_reddit_place_sentiments["reviewer"].where(
    df_reddit_place_sentiments["reviewer"].notna()
    & (df_reddit_place_sentiments["reviewer"] != ""), "None")

print(f"\n📍 {len(df_reddit_place_sentiments):,} unique places")
print(f"   place_type coverage: "
      f"{df_reddit_place_sentiments["place_type"].notna().sum()} / "
      f"{len(df_reddit_place_sentiments)} labelled")
print(f"\nTop types:\n{df_reddit_place_sentiments["place_type"].value_counts(dropna=False).head(8).to_string()}")

(df_reddit_place_sentiments.sample(5, random_state=42)
 .style.set_properties(subset=["sentences"], **{"min-width": "420px", "white-space": "normal"}))

---

## 2 The Pipeline — A Critical Review

Before we visualize anything, it is worth stepping back to review what each stage of the pipeline actually did — and where it could have gone wrong.

### Lesson 3: Data Cleaning
You loaded raw Reddit posts and split them into individual sentences. You cleaned up encoding issues, removed very short strings, and filtered out noise. This early stage determines what the data will look like later.

> 👉 **Note:** *Sentence splitting is imperfect. A sentence that spans a line break or uses non-standard punctuation may have been cut in the wrong place, which would mislead the sentiment model.*

### Lesson 4: Location Extraction
You ran two NER (Named Entity Recognition) models — spaCy and a transformer-based tagger — to identify place names in each sentence, then used a geoparser to convert those names to coordinates.

> 👉 **Note:** *NER models confuse place names with other entities (people, organizations, common words). The geoparser resolves ambiguous names by population rank and other weights, which means "London" will almost always map to the UK even if another location is closer and the author meant something else. You corrected some of these manually — but not all of them.*

### Lesson 5: Sentiment Analysis
You ran VADER (rule-based, fast) and RoBERTa (transformer, context-aware) on each sentence and compared their outputs. You found contradictions — sentences where the two models disagreed — and used them to understand each model's blind spots.

> 👉 **Note:** *Both models were trained on general social media text. Reddit language, irony, sarcasm, and in-group references may confuse either model. Sentiment scores are averages across sentences — a place mentioned once in a very negative post will look "negative" even if 99% of posts about it are neutral.*


---

## 3 Fuzzy Data and Critical Design Decisions

There is a law of diminishing returns when cleaning up the geoparsed data. At some point, it is no longer worth parsing each individual location and you have to accept that some of the data will be inaccurate. Likewise, the sentiment analyzer is going to give a rough estimate, but will not give a 100% accurate picture of all the emotions around each location. There is an inherent fuzziness about the way humans talk about locations and the emotions they attach to them. Indeed, the phrase "I love JMU!" is an emotion about a location that seems very definite, but even here it may only be one particular space in JMU you love and at one particular time. Since the data is so fuzzy we have to be very critical in how we visualize it.

Modern mapping technologies like Plotly and Mapbox allow us to create maps very quickly. If we do not consider the underlying data, we can come up with two radically different representations. Consider the two maps below. Each takes data we created about JMU and puts it on a map. Consider what conclusion you would draw from each map.


In [ ]:
# ── Data prep ────────────────────────────────────────────────────────────
df_A = df_reddit_place_sentiments.copy()                          # Map A: everything
df_B = df_reddit_place_sentiments[
    df_reddit_place_sentiments["location_count"] >= 5             # Map B: high-confidence only
].copy()

# ── Map A: design choices that read as "students are negative" ───────────
#
# The key manipulation is the MISSING color_continuous_midpoint.
# Plotly anchors the neutral (yellow) color at the midpoint of the
# data RANGE, not at zero.  If sentiment spans [-0.3, +0.6], the midpoint
# is +0.15 — so ANY place with sentiment below +0.15 (including mildly
# positive ones) renders red-to-yellow on the scale.

fig_A = px.scatter_map(
    df_A,
    lat="latitude",
    lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    labels={
        "avg_roberta_compound": "Sentiment",
        "sentiment_category": "Sentiment Intensity",
        "location_count": "Count",
        "reviewer": "Reviewer",
    },
    custom_data=["place","avg_roberta_compound","location_count", "reviewer"],
    color_continuous_scale="RdYlGn",
    # ← NO color_continuous_midpoint: neutral color drifts to the data midpoint
    size_max=70,  # ← one place will dominate visually
    map_style="carto-darkmatter",  # ← dark basemap, ominous tone
    center={"lat": 39.0, "lon": -89.0},  # ← national framing
    zoom=3,  # ← geoparser errors fully visible
    height=620,
    title="<b>Map A</b>",
)
fig_A.update_layout(margin={"r": 0, "t": 90, "l": 0, "b": 0})
fig_A.update_traces(
    hovertemplate="<b>%{customdata[0]}</b><br>"
    + "Sentiment: %{customdata[1]:.2f}<br>"
    + "Count: %{customdata[2]} <br>"
    + "Reviewer: %{customdata[3]}"
    + "<extra></extra>",
)
fig_A.show()

# ── Map B: design choices that read as "sentiment is mostly positive" ────

NEUTRAL_THRESHOLD = 0.10   # generous neutral band — ±0.10 around zero

bins       = [-float("inf"), -NEUTRAL_THRESHOLD, NEUTRAL_THRESHOLD, float("inf")]
cat_labels = ["Negative", "Neutral", "Positive"]
df_B["sentiment_category"] = pd.cut(
    df_B["avg_roberta_compound"], bins=bins, labels=cat_labels
)

# Quantile size classification: equal number of places per class.
# labels=False gives 0-indexed integer codes; +1 shifts to 1-indexed.
# This avoids a ValueError when duplicates='drop' reduces the bin count
# below the fixed labels list.
df_B["size_class"] = (
    pd.qcut(df_B["location_count"], q=4, labels=False, duplicates="drop")
    .astype(float) + 1
)

fig_B = px.scatter_map(
    df_B,
    lat="latitude",
    lon="longitude",
    size="size_class",
    color="sentiment_category",
    hover_name="place",
    labels={
        "avg_roberta_compound": "Sentiment",
        "sentiment_category": "Sentiment Intensity",
        "location_count": "Count",
        "reviewer": "Reviewer",
    },
    custom_data=["place", "avg_roberta_compound", "location_count", "reviewer"],
    color_discrete_map={
        "Negative": "#d62728",
        "Neutral": "#aec7e8",
        "Positive": "#2ca02c",
    },
    category_orders={"sentiment_category": cat_labels},
    size_max=15,
    map_style="carto-positron",  # ← clean, neutral basemap
    center={"lat": 38.4, "lon": -79.0},  # ← Shenandoah Valley / campus context
    zoom=7,  # ← regional framing
    height=620,
    title="<b>Map B</b>",
)
fig_B.update_layout(margin={"r": 0, "t": 90, "l": 0, "b": 0})
fig_A.update_traces(
    hovertemplate="<b>%{customdata[0]}</b><br>"
    + "Sentiment: %{customdata[1]:.2f}<br>"
    + "Count: %{customdata[2]} <br>"
    + "Reviewer: %{customdata[3]}"
    + "<extra></extra>",
)
fig_B.show()

> 💡 **Reflection:** 
> - What does each map tell you? 
> - Why do you reach that conclusion? 
> - What are some visual distortions that each introduces?
> &nbsp;

---

## 4 Map Design

Map design is a highly specialized field in cartography with its own long history. For this lesson, we will cover three major design principles: symbolization, color theory, and scale. Each of these principles helps tell a particular story. 

- **Symbolization** - determines what marker or symbol you use to represent your data. This covers the number of symbols, their relative size, shape, and which data points are worth showing at all.

- **Color theory** - determines how you are going to differentiate different elements on the map through your color choices. This includes how you distinguish between symbols, but also how you contrast the base map style with the data projected onto it. One major consideration for color theory is always accessibility. How do you use colors effectively so that a maximum number of users can understand your story?

- **Scale** - determines the zoom and viewport of your map. This may seem simple, but you have to consider the conceptual scale of your story. Are you telling a story about Harrisonburg, Virginia, the US, or even the entire world? Determining the scale of the story makes the zoom far more consequential.

Each of these principles has been broken down into mini-units on each element below. **Run each code cell, compare the versions, and answer the reflection questions before moving on.** Do not proceed to the design brief until you have worked through all seven.

| Principle | # | Design Decision | The core question |
|---|---|---|---|
| **Symbolization** | 1 | **Filtering threshold** | Which places are worth showing? |
| &nbsp; | 2 | **Place type filter** | Should Country and City sentiment appear on the same map? |
| &nbsp; | 3a | **Bubble size encoding** | Linear, square root, or uniform — which is honest? |
| &nbsp; | 3b | **Size classification** | Equal interval, quantile, or Jenks natural breaks? |
| &nbsp; | 4 | **Sentiment bucketing** | Continuous gradient or discrete categories? |
| **Color theory** | 5 | **Color scale** | Which scale is honest *and* accessible? |
| &nbsp; | 6 | **Base map style** | What tone and context does the basemap set? |
| **Scale** | 7 | **Zoom & viewport** | What context do you show — and what do you hide? |


### Decision 1: Filtering Threshold

The data is very wide ranging. Some locations appear only once while others appear many times (i.e. Harrisonburg). While a "true" map would show all the data, displaying each individual location  can be visually distracting and make it much harder to see patterns. You can filter out this noise by adjusting the minimum count for each location. Thus, if a location only appears 2 times it might not be all that relevant. 

The tricky thing is that there is no concrete rule for picking a threshold. Instead, it depends on a host of factors. If you are doing an analysis of the buildings mentioned in each data set they may not be mentioned all that often so a lower threshold is necessary. Alternatively, if you are doing an analysis of cities mentioned, the threshold may be higher because cities tend to be named more often. The sweet spot is one that shows enough data to tell the story without crowding the map, while also not erasing data that could contradict your story. 

You will have to justify your threshold in your project, so consider this carefully.

In the code below, you can change the `min_count` variable. Play around with the value to see how much clutter you can remove. Be sure to zoom in and out to take a larger view.


In [ ]:
# Decision 1: Filtering threshold — how much data is enough?

# ── Decision 1: change this value and re-run ─────────────────────────────────
MIN_COUNT = 3  # ← minimum number of posts for a place to appear on the map

# ── Data preparation ──────────────────────────────────────────────────────────
df_work = df_reddit_place_sentiments[
    df_reddit_place_sentiments["location_count"] >= MIN_COUNT
].copy()

print(f"── min_count ≥ {MIN_COUNT}: {len(df_work)} locations, "
      f"{df_work["location_count"].sum()} total posts ──")

fig = px.scatter_map(
    df_work,
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    hover_data={
        "location_count": True,
        "avg_roberta_compound": ":.3f",
        "latitude": False,
        "longitude": False,
    },
    color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
    size_max=20,
    map_style="carto-positron",
    center={
        "lat": 37.5,
        "lon": -78.0,
    },
    zoom=6, height=450,
    title=f"Decision 1 — min post count ≥ {MIN_COUNT}  →  {len(df_work)} locations"
)
fig.update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    },
    coloraxis_showscale=False,
)
fig.show()

> 💡 **Reflection:** What do you think is the "sweet spot" for eliminating the minimum count? Why do you think that is?


### Decision 2: Place Type Filter

Locations exist at different scales and are therefore different in kind. The location "United States" is a country, while "Harrisonburg" is a city. Technically, they both exist on the same map, but if you are trying to tell a story about how JMU students feel about their community country-level information might not be all that relevant. Conversely, if you are considering how JMU feels about other places in Virginia then knowing how students feel about D-Hall is not all that relevant.

In lesson 4 we cleaned up the geoparsed `place_types` that made these distinctions. One way we can reduce map distortion is making sure that all the symbols were are looking at are the ones we need for our particular story.

We can filter for specific types by adding those types to the `place_types` list.

```
place_types = ['City', 'Building']
```

This filter will only show Cities and Buildings. The options include:

`'Country', 'State', 'City', 'Neighborhood','Building', , 'University', 'Road', 'Region','Natural Feature` 

In [ ]:
# Decision 2: Place type filter — which geographic scales belong in this story?

# ── Parameters from previous decisions ───────────────────────────────────────
MIN_COUNT = 3  # Decision 1

# ── Decision 2: change this value and re-run ─────────────────────────────────
PLACE_TYPES = ["City", "Building", "University", "Neighborhood"]  # ← set to None to show all types
# Named options: 'Country', 'State', 'Region', 'City', 'Neighborhood',
#                'University', 'Road', 'Building', 'Natural Feature'

# ── Data preparation ──────────────────────────────────────────────────────────
if PLACE_TYPES is None:
    df_work = df_reddit_place_sentiments[
        df_reddit_place_sentiments["location_count"] >= MIN_COUNT
    ].copy()
else:
    df_work = df_reddit_place_sentiments[
        (df_reddit_place_sentiments["location_count"] >= MIN_COUNT) &
        df_reddit_place_sentiments["place_type"].isin(PLACE_TYPES)
    ].copy()

type_counts = df_work["place_type"].value_counts(dropna=False).to_dict()
print(f"── min_count ≥ {MIN_COUNT}, place_types = {PLACE_TYPES}: "
      f"{len(df_work)} places  {type_counts} ──")

fig = px.scatter_map(
    df_work,
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    hover_data={
        "place_type": True,
        "location_count": True,
        "avg_roberta_compound": ":.3f",
        "latitude": False,
        "longitude": False,
    },
    color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
    size_max=20,
    map_style="carto-positron",
    center={
        "lat": 37.5,
        "lon": -78.0,
    },
    zoom=6, height=450,
    title=f"Decision 2 — place_types = {PLACE_TYPES}  →  {len(df_work)} locations"
)
fig.update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    },
    coloraxis_showscale=False,
)
fig.show()

> 💡 **Reflection:** How does only showing the Cities and the Buildings affect what data is seen? Does reducing this clutter start to tell a different story?

### Decision 3: Bubble Size

A bubble map uses the size of the bubble to denote relative importance. Generally, the largest bubbles catch the attention of viewers first and form the center of the story you are trying to tell. Selecting the appropriate bubble sizing is difficult because of how humans perceive relative size differences. Take a look at the bubbles below. 



<img src="../lesson_assets/images/bubble-size-difference.png" style="height: 200px; vertical-align: middle;">
<br>

It is easy to see that one bubble is larger than the other, but it is much harder to know the order of magnitude or *how*  much larger the largest bubble is than the others.

#### 3a: Size Encoding — Linear, Square Root, or Uniform?

The most common approach to dealing with bubble size is to set the size directly to the number of times a particular variable occurs at a location. This is usually referred to as the `count`. In our case, this is every time the location is mentioned, but it could equally well be something like a crime statistic or the total number of people at a location. The three maps below compare what happens when you try each encoding strategy:

| Encoding | What it emphasizes | Risk |
|---|---|---|
| `size = raw count` (linear) | Frequency of discussion | One outlier can overwhelm everything else |
| `size = √count` (square root) | Frequency, compressed | Scale is less intuitive; readers may not know how to decode it |
| Uniform size | Nothing — only color speaks | Makes it harder to see where the data is sparse vs. rich |

None of these options is fully satisfactory. Raw count lets a single heavily-discussed location visually dominate the map. Square root compresses the scale, but the encoding is harder for a reader to interpret intuitively. Uniform size discards frequency information entirely, leaving only color to carry the argument.

The better approach is to **classify** counts into a small number of discrete size classes — grouping similar-count locations together so that relative importance comes through without one outlier drowning out everything else. That is what **3b** covers below.


In [ ]:
# Decision 3a: Bubble size encoding — what does SIZE encode?

# ── Parameters from previous decisions ───────────────────────────────────────
MIN_COUNT   = 3                                              # Decision 1
PLACE_TYPES = ["City", "Building", "University", "Neighborhood"]  # Decision 2

# ── Data preparation ──────────────────────────────────────────────────────────
if PLACE_TYPES is not None:
    df_work = df_reddit_place_sentiments[
        (df_reddit_place_sentiments["location_count"] >= MIN_COUNT) &
        df_reddit_place_sentiments["place_type"].isin(PLACE_TYPES)
    ].copy()
else:
    df_work = df_reddit_place_sentiments[
        df_reddit_place_sentiments["location_count"] >= MIN_COUNT
    ].copy()

df_work["sqrt_count"] = np.sqrt(df_work["location_count"])
df_work["uniform"]    = 1

base = {
    "lat": "latitude",
    "lon": "longitude",
    "color": "avg_roberta_compound",
    "hover_name": "place",
    "hover_data": {
        "location_count": True,
        "avg_roberta_compound": ":.3f",
        "latitude": False,
        "longitude": False,
    },
    "color_continuous_scale": "RdYlGn",
    "color_continuous_midpoint": 0,
    "map_style": "carto-positron",
    "center": {
        "lat": 37.5,
        "lon": -78.0,
    },
    "zoom": 6,
    "height": 460,
}

for size_col, size_max, label in [
    ("location_count", 25, "A — size = raw count (linear): popular places visually dominate"),
    ("sqrt_count",     14, "B — size = √count: compressed scale, less outlier dominance"),
    ("uniform",         5, "C — uniform size: only color carries information"),
]:
    fig = px.scatter_map(df_work, size=size_col, size_max=size_max, title=label, **base)
    fig.update_layout(
        margin={
            "r": 0,
            "t": 50,
            "l": 0,
            "b": 0,
        },
        coloraxis_showscale=False,
    )
    fig.show()

> 📊 **Output:** In the first map Harrisonburg dominates the entire space, while in the second map locations with a lower count suddenly gain more prominence. Using a uniform size like the last map might make sense if all the numbers are relatively similar, but there is a significant range here.


#### 3b: Size Classification — Equal Interval, Quantile, or Jenks?

Since continuous size encodings are all unsatisfactory in different ways, the practical solution is to **bucket** post counts into a small number of discrete size classes. All locations in the same class get the same bubble size. This groups relative importance with relative visual weight, without letting one outlier stretch the entire scale.

Our Reddit post counts are almost always **right-skewed**: a handful of well-known places accumulate dozens or hundreds of mentions, while the majority of locations appear only once or twice. This is because the thread is about JMU, so you would expect most of the conversation to be about Harrisonburg and the surrounding area. The three standard classification methods handle that skew in very different ways:

| Method | How it works | Best when... |
|---|---|---|
| **Equal interval** | Divides the count *range* into bins of equal width (e.g., 0–25, 25–50, 50–75, 75–100) | Data is roughly uniformly distributed — rare for geographic counts |
| **Quantile** | Divides the *sorted data* so each bin holds the same number of places (e.g., 25th / 50th / 75th percentile breaks) | You want every size class to appear equally often on the map |
| **Jenks natural breaks** | Finds the threshold values that **minimize within-class variance** — breaks fall at genuine gaps in the data | Data has natural clusters (almost always true for post-count data) |

The chart below shows how each method divides the JMU post-count distribution. Vertical lines mark the breakpoints; the colored band shows the width of each class.

<img src="../lesson_assets/images/classification-methods.png" style="max-width: 580px; display: block; margin: 1em 0;">

**Why equal interval usually fails here:** If most places have 1–5 mentions and one has 200, all but one place fall into the lowest bin. The largest bin contains a single point — which defeats the purpose of classification entirely.

**Why quantile can also mislead:** Because more than two-thirds of places appear only once, quantile classification collapses to just two bins — the top chart above shows this clearly. Most of the data range is assigned to one enormous class.

**Jenks is usually the right choice** for this data. It finds where the *real* gaps are — for example, it might identify {1–3, 4–10, 11–40, 41+} as the natural clusters rather than dividing the range arithmetically.

> 💡 **Questions to consider:**
> - After running the cell, how many places land in the largest equal-interval bin?
> - Where do the Jenks break points fall? Do those values correspond to any intuitive groupings?
> - Does the visual differentiation between small, medium, and large bubbles improve with Jenks?


In [ ]:
# Decision 3b: Size classification — equal interval, quantile, Jenks natural breaks

import mapclassify

# ── Parameters from previous decisions ───────────────────────────────────────
MIN_COUNT   = 3                                              # Decision 1
PLACE_TYPES = ["City", "Building", "University", "Neighborhood"]  # Decision 2

# ── Decision 3b: change this value and re-run ────────────────────────────────
N_SIZE_CLASSES = 4  # ← number of size classes; try 3 or 5

# ── Data preparation ──────────────────────────────────────────────────────────
if PLACE_TYPES is not None:
    df_work = df_reddit_place_sentiments[
        (df_reddit_place_sentiments["location_count"] >= MIN_COUNT) &
        df_reddit_place_sentiments["place_type"].isin(PLACE_TYPES)
    ].copy()
else:
    df_work = df_reddit_place_sentiments[
        df_reddit_place_sentiments["location_count"] >= MIN_COUNT
    ].copy()

counts = df_work["location_count"]

# Method A: Equal interval — bins of equal WIDTH
df_ei = df_work.copy()
df_ei["size_class"] = pd.cut(counts, bins=N_SIZE_CLASSES, labels=range(1, N_SIZE_CLASSES + 1)).astype(float)

# Method B: Quantile — bins of equal COUNT (same number of places per class)
# labels=False returns 0-indexed integer codes; +1 shifts to 1-indexed.
# This avoids a ValueError when duplicates='drop' reduces the bin count
# below the fixed labels list.
df_qt = df_work.copy()
df_qt["size_class"] = (
    pd.qcut(counts, q=N_SIZE_CLASSES, labels=False, duplicates="drop")
    .astype(float) + 1
)

# Method C: Jenks natural breaks — bins at genuine gaps in the data
try:
    jnb = mapclassify.NaturalBreaks(counts.values, k=N_SIZE_CLASSES)
    df_jnb = df_work.copy()
    df_jnb["size_class"] = (jnb.yb + 1).astype(float)
    c_label = "C — Jenks: breaks at genuine gaps in the data (minimizes within-class variance)"
except ImportError:
    print("\n⚠️  mapclassify not installed. Run:  pip install mapclassify")
    print("   Showing Quantile as fallback for C.")
    df_jnb = df_qt.copy()
    c_label = "C — Jenks unavailable (mapclassify not installed); showing Quantile"

base = {
    "lat": "latitude",
    "lon": "longitude",
    "color": "avg_roberta_compound",
    "hover_name": "place",
    "hover_data": {
        "location_count": True,
        "size_class": False,
        "avg_roberta_compound": ":.3f",
        "latitude": False,
        "longitude": False,
    },
    "color_continuous_scale": "RdYlGn",
    "color_continuous_midpoint": 0,
    "size_max": 18,
    "map_style": "carto-positron",
    "center": {
        "lat": 37.5,
        "lon": -78.0,
    },
    "zoom": 6,
    "height": 460,
}

for label, df_method in [
    ("A — Equal Interval: uniform bin width; dominated by skewed outliers", df_ei),
    ("B — Quantile: equal number of places per class; breaks may fall at arbitrary counts", df_qt),
    (c_label, df_jnb),
]:
    fig = px.scatter_map(df_method, size="size_class", title=label, **base)
    fig.update_layout(
        margin={
            "r": 0,
            "t": 55,
            "l": 0,
            "b": 0,
        },
        coloraxis_showscale=False,
    )
    fig.show()

### Decision 4: Sentiment Bucketing — Continuous or Classified?

Decision 3 showed that minor size differences between bubbles are perceptually invisible. As such, a bubble representing 12 posts looks almost identical to one representing 15. The solution was to **classify** counts into a small number of discrete size classes so that the visual differences are large enough to register.

The same perceptual problem applies to **color**. RoBERTa produces a score from roughly −1 to +1, and a continuous gradient maps every decimal of that range to a slightly different shade. But the human eye cannot reliably distinguish between −0.12 and −0.18 on a red-to-green scale. The gradient implies a precision that neither the viewer nor the underlying data can support.

The fix is the same: classify the sentiment scores into a small number of discrete color buckets so each category is visually unambiguous.

**The threshold problem with fixed categories:**
The simplest approach — Negative / Neutral / Positive with a hand-chosen ±threshold — is easy to read, but the threshold is itself an argument. A threshold of ±0.05 calls most places "Neutral." A threshold of ±0.20 calls most places "Negative" or "Positive." There is no objectively correct cutoff.

**The data-driven alternative — Jenks natural breaks:**
For exactly the same reasons as Decision 3b, the best method for determining color buckets is **Jenks natural breaks**. Instead of imposing a fixed threshold, Jenks finds where the *genuine gaps* are in the sentiment distribution — the values where the data naturally clusters. The break points emerge from the data rather than from an assumption.

You can also control how many buckets to use. Fewer buckets (3–4) produce a legible map but collapse variation. More buckets (6–8) preserve more distinction but may start to look like a continuous scale again.

> 💡 **Reflection:** Compare versions B and C below. Does the fixed ±threshold place the breaks where the data actually clusters? What does Jenks reveal that the fixed approach hides?

### ✍️ 4.1 Critical Activity - Testing Sentiment Buckets

Change `N_COLOR_BUCKETS` to 3, then to 7. At what point does increasing the number of buckets stop adding useful information?

In [ ]:
# Decision 4: Sentiment bucketing — continuous, fixed threshold, or Jenks?

import mapclassify
import plotly.colors as pc

# ── Parameters from previous decisions ───────────────────────────────────────
MIN_COUNT      = 3                                              # Decision 1
PLACE_TYPES    = ["City", "Building", "University", "Neighborhood"]  # Decision 2
N_SIZE_CLASSES = 4                                              # Decision 3b: Jenks size classes

# ── Decision 4: change these values and re-run ───────────────────────────────
NEUTRAL_THRESHOLD = 0.05  # ← threshold for Version B (fixed categories)
N_COLOR_BUCKETS   = 7     # ← number of color buckets for Version C (Jenks)

# ── Data preparation ──────────────────────────────────────────────────────────
if PLACE_TYPES is not None:
    df_work = df_reddit_place_sentiments[
        (df_reddit_place_sentiments["location_count"] >= MIN_COUNT) &
        df_reddit_place_sentiments["place_type"].isin(PLACE_TYPES)
    ].copy()
else:
    df_work = df_reddit_place_sentiments[
        df_reddit_place_sentiments["location_count"] >= MIN_COUNT
    ].copy()

# Apply Jenks size classification (Decision 3b)
_jnb_s = mapclassify.NaturalBreaks(df_work["location_count"].values, k=N_SIZE_CLASSES)
df_work["size_class"] = (_jnb_s.yb + 1).astype(float)

scores = df_work["avg_roberta_compound"]

base = {
    "lat": "latitude",
    "lon": "longitude",
    "size": "size_class",
    "hover_name": "place",
    "hover_data": {
        "avg_roberta_compound": ":.3f",
        "location_count": True,
        "size_class": False,
        "latitude": False,
        "longitude": False,
    },
    "size_max": 18,
    "map_style": "carto-positron",
    "center": {
        "lat": 37.5,
        "lon": -78.0,
    },
    "zoom": 6,
    "height": 490,
}

# ── Version A: continuous ─────────────────────────────────────────────────────
fig_A = px.scatter_map(
    df_work, color="avg_roberta_compound",
    color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
    title="A — Continuous: precise, but minor shade differences are imperceptible",
    **base
)
fig_A.update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
)
fig_A.show()

# ── Version B: fixed ±threshold ───────────────────────────────────────────────
cat_labels = ["Negative", "Neutral", "Positive"]
df_work["bucket_fixed"] = pd.cut(
    scores,
    bins=[-float("inf"), -NEUTRAL_THRESHOLD, NEUTRAL_THRESHOLD, float("inf")],
    labels=cat_labels
)

fig_B = px.scatter_map(
    df_work, color="bucket_fixed",
    color_discrete_map={
        "Negative": "#d62728",
        "Neutral": "#aec7e8",
        "Positive": "#2ca02c",
    },
    category_orders={
        "bucket_fixed": cat_labels,
    },
    title=f"B — Fixed threshold ±{NEUTRAL_THRESHOLD}: readable, but the threshold is an arbitrary claim",
    **base
)
fig_B.update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
)
fig_B.show()

# ── Version C: Jenks natural breaks ───────────────────────────────────────────
_jnb_c  = mapclassify.NaturalBreaks(scores.values, k=N_COLOR_BUCKETS)
_breaks = _jnb_c.bins
_lo     = scores.min()
_labels = []
for _hi in _breaks:
    _labels.append(f"{_lo:.2f} to {_hi:.2f}")
    _lo = _hi

df_work["bucket_jenks"] = pd.cut(scores, bins=[-float("inf")] + list(_breaks), labels=_labels)
_palette   = pc.sample_colorscale("RdYlGn", [i / (N_COLOR_BUCKETS - 1) for i in range(N_COLOR_BUCKETS)])
_color_map = {label: color for label, color in zip(_labels, _palette)}

fig_C = px.scatter_map(
    df_work, color="bucket_jenks",
    color_discrete_map=_color_map,
    category_orders={
        "bucket_jenks": _labels,
    },
    title=f"C — Jenks natural breaks ({N_COLOR_BUCKETS} classes): breaks fall at genuine gaps in the sentiment distribution",
    **base
)
fig_C.update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
)
fig_C.show()

### Decision 5: Color Scale

Color is the most powerful — and most easily manipulated — variable on a sentiment map.

**Diverging vs. sequential:**
Diverging scales (red↔green, red↔blue) are appropriate when zero is a meaningful midpoint and values exist on both sides. Setting `color_continuous_midpoint=0` anchors the neutral color at zero. Sequential scales (light→dark) work better when all values fall on one side and are not appropriate here unless you are mapping only one sentiment category.

**Cultural associations and accessibility:**
- **Red/green (RdYlGn)** is immediately intuitive because red=danger and green=safe are deeply ingrained. But red-green color blindness affects roughly 8% of men — for those readers, the entire argument of your map is invisible.
- **Red/blue (RdBu)** is more accessible and carries less cultural baggage about "good" and "bad."
- **Viridis** is fully colorblind-safe and perceptually uniform, but loses the positive/negative framing — every value looks like a point on a temperature scale rather than an emotional one.

**The midpoint is a claim:**
A diverging scale with its midpoint at 0 says: "zero means neutral." If your data skews positive (most scores above zero), shifting the midpoint upward could make the map look more balanced — or more negative — without changing a single data value.

> 💡 **Questions to consider:**
> - Who cannot read the RdYlGn map? What is your responsibility to that reader?
> - At what point does choosing a flattering color scale become misleading?
> - Should "neutral" be at the center of the color scale, or at the center of your data's range?


In [ ]:
# Decision 5: Color scale — how does palette choice shape the emotional read of the map?

import mapclassify
import plotly.colors as pc

# ── Parameters from previous decisions ───────────────────────────────────────
MIN_COUNT       = 3                                              # Decision 1
PLACE_TYPES     = ["City", "Building", "University", "Neighborhood"]  # Decision 2
N_SIZE_CLASSES  = 4                                              # Decision 3b: Jenks size classes
N_COLOR_BUCKETS = 5                                              # Decision 4: Jenks color buckets

# ── Decision 5: change this value and re-run ─────────────────────────────────
COLOR_SCALE = "RdYlGn"  # ← try 'RdBu', 'Viridis', 'Spectral'

# ── Data preparation ──────────────────────────────────────────────────────────
if PLACE_TYPES is not None:
    df_work = df_reddit_place_sentiments[
        (df_reddit_place_sentiments["location_count"] >= MIN_COUNT) &
        df_reddit_place_sentiments["place_type"].isin(PLACE_TYPES)
    ].copy()
else:
    df_work = df_reddit_place_sentiments[
        df_reddit_place_sentiments["location_count"] >= MIN_COUNT
    ].copy()

# Apply Jenks size classification (Decision 3b)
_jnb_s = mapclassify.NaturalBreaks(df_work["location_count"].values, k=N_SIZE_CLASSES)
df_work["size_class"] = (_jnb_s.yb + 1).astype(float)

# Apply Jenks color classification (Decision 4)
scores  = df_work["avg_roberta_compound"]
_jnb_c  = mapclassify.NaturalBreaks(scores.values, k=N_COLOR_BUCKETS)
_breaks = _jnb_c.bins
_lo     = scores.min()
_labels = []
for _hi in _breaks:
    _labels.append(f"{_lo:.2f} to {_hi:.2f}")
    _lo = _hi
df_work["color_class"] = pd.cut(scores, bins=[-float("inf")] + list(_breaks), labels=_labels)


base = {
    "lat": "latitude",
    "lon": "longitude",
    "size": "size_class",
    "color": "color_class",
    "hover_name": "place",
    "hover_data": {
        "avg_roberta_compound": ":.3f",
        "location_count": True,
        "size_class": False,
        "color_class": False,
        "latitude": False,
        "longitude": False,
    },
    "category_orders": {
        "color_class": _labels,
    },
    "size_max": 18,
    "map_style": "carto-positron",
    "center": {
        "lat": 37.5,
        "lon": -78.0,
    },
    "zoom": 6,
    "height": 460,
}

for title, scale in [
    ("RdYlGn  — default diverging (red=bad, green=good, accessibility issues)", "RdYlGn"),
    ("RdBu  — diverging, more accessible, less culturally loaded",            "RdBu"),
    ("Viridis — colorblind-safe, perceptually uniform, but loses pos/neg frame","Viridis"),
    ("Spectral — high-contrast diverging",                                      "Spectral"),
]:
    _palette   = pc.sample_colorscale(scale, [i / (N_COLOR_BUCKETS - 1) for i in range(N_COLOR_BUCKETS)])
    _color_map = {label: color for label, color in zip(_labels, _palette)}
    fig = px.scatter_map(df_work, color_discrete_map=_color_map, title=title, **base)
    fig.update_layout(
        margin={
            "r": 0,
            "t": 50,
            "l": 0,
            "b": 0,
        },
       legend_traceorder="reversed"
    )
    fig.show()

### Decision 6: Base Map Style

The base map is the layer on which the data is rendered. It not only provides visual context but also makes implicit claims about geographic precision that the geoparsed data may not support.

| Style | Message it sends | Risk |
|---|---|---|
| Minimal / light (carto-positron) | Focus is on the data; geography is background | Strips context that might help readers orient themselves |
| Dark (carto-darkmatter) | High contrast; sentiment colors pop | Can make the map feel ominous regardless of the actual values |
| Full street map (OpenStreetMap) | Rich geographic context | Roads and labels compete with the data; implies more precision than geoparsing provides |

> 💡 **Questions to consider:**
> - Does a dark base map change how you *feel* about the sentiment data even before reading a value?
> - At what point does a stylistic choice become a rhetorical one?


In [ ]:
# Decision 6: Base map style — what does the background communicate?

import mapclassify
import plotly.colors as pc

# ── Parameters from previous decisions ───────────────────────────────────────
MIN_COUNT       = 3                                              # Decision 1
PLACE_TYPES     = ["City", "Building", "University", "Neighborhood"]  # Decision 2
N_SIZE_CLASSES  = 4                                              # Decision 3b: Jenks size classes
N_COLOR_BUCKETS = 5                                              # Decision 4: Jenks color buckets
COLOR_SCALE     = "RdYlGn"                                      # Decision 5: color scale

# ── Decision 6: change this value and re-run ─────────────────────────────────
MAP_STYLE = "carto-positron"  # ← try 'carto-darkmatter', 'open-street-map'

# ── Data preparation ──────────────────────────────────────────────────────────
if PLACE_TYPES is not None:
    df_work = df_reddit_place_sentiments[
        (df_reddit_place_sentiments["location_count"] >= MIN_COUNT) &
        df_reddit_place_sentiments["place_type"].isin(PLACE_TYPES)
    ].copy()
else:
    df_work = df_reddit_place_sentiments[
        df_reddit_place_sentiments["location_count"] >= MIN_COUNT
    ].copy()

# Apply Jenks size classification (Decision 3b)
_jnb_s = mapclassify.NaturalBreaks(df_work["location_count"].values, k=N_SIZE_CLASSES)
df_work["size_class"] = (_jnb_s.yb + 1).astype(float)

# Apply Jenks color classification (Decision 4)
scores  = df_work["avg_roberta_compound"]
_jnb_c  = mapclassify.NaturalBreaks(scores.values, k=N_COLOR_BUCKETS)
_breaks = _jnb_c.bins
_lo     = scores.min()
_labels = []
for _hi in _breaks:
    _labels.append(f"{_lo:.2f} to {_hi:.2f}")
    _lo = _hi
df_work["color_class"] = pd.cut(scores, bins=[-float("inf")] + list(_breaks), labels=_labels)
_palette   = pc.sample_colorscale(COLOR_SCALE, [i / (N_COLOR_BUCKETS - 1) for i in range(N_COLOR_BUCKETS)])
_color_map = {label: color for label, color in zip(_labels, _palette)}

base = {
    "lat": "latitude",
    "lon": "longitude",
    "size": "size_class",
    "color": "color_class",
    "hover_name": "place",
    "hover_data": {
        "avg_roberta_compound": ":.3f",
        "location_count": True,
        "size_class": False,
        "color_class": False,
        "latitude": False,
        "longitude": False,
    },
    "color_discrete_map": _color_map,
    "category_orders": {
        "color_class": _labels,
    },
    "size_max": 18,
    "height": 460,
}

for label, style in [
    ("Light — carto-positron (minimal, data-focused)",    "carto-positron"),
    ("Dark — carto-darkmatter (high contrast, dramatic)", "carto-darkmatter"),
    ("Standard — open-street-map (maximum context)",      "open-street-map"),
]:
    fig = px.scatter_map(
        df_work, map_style=style,
        center={
            "lat": 37.5,
            "lon": -78.0,
        },
        zoom=6,
        title=label, **base
    )
    fig.update_layout(
        margin={
            "r": 0,
            "t": 50,
            "l": 0,
            "b": 0,
        }
    )
    fig.show()

### Decision 7: Zoom & Viewport

Although users can manipulate the zoom on a map manually, the opening view will always make a statement about what is important. Zoom level determines what is *in frame* and what is cut off. Zoom out to national scale and you will see geoparser errors: posts resolved to Paris, London, or cities in Asia. Zoom into Harrisonburg and those errors are out of frame — which makes the map cleaner, but also hides the fact that those errors exist.

> 💡 **Questions to consider:**
> - At national zoom, do any locations look obviously wrong?
> - Is choosing a tight zoom that hides geoparser errors honest? What should you disclose in a caption?
> - What geographic extent best serves your research question?


In [ ]:
# Decision 7: Zoom & viewport — what story does your framing tell?

import mapclassify
import plotly.colors as pc

# ── Parameters from previous decisions ───────────────────────────────────────
MIN_COUNT       = 3                                              # Decision 1
PLACE_TYPES     = ["City", "Building", "University", "Neighborhood"]  # Decision 2
N_SIZE_CLASSES  = 4                                              # Decision 3b: Jenks size classes
N_COLOR_BUCKETS = 5                                              # Decision 4: Jenks color buckets
COLOR_SCALE     = "RdYlGn"                                      # Decision 5: color scale
MAP_STYLE       = "carto-positron"                              # Decision 6: base map style

# ── Decision 7: change these values and re-run ───────────────────────────────
CENTER = {  # ← map center
    "lat": 37.5,
    "lon": -78.0,
}
ZOOM   = 6                            # ← zoom level (3 = national, 11 = campus)

# ── Data preparation ──────────────────────────────────────────────────────────
if PLACE_TYPES is not None:
    df_work = df_reddit_place_sentiments[
        (df_reddit_place_sentiments["location_count"] >= MIN_COUNT) &
        df_reddit_place_sentiments["place_type"].isin(PLACE_TYPES)
    ].copy()
else:
    df_work = df_reddit_place_sentiments[
        df_reddit_place_sentiments["location_count"] >= MIN_COUNT
    ].copy()

# Apply Jenks size classification (Decision 3b)
_jnb_s = mapclassify.NaturalBreaks(df_work["location_count"].values, k=N_SIZE_CLASSES)
df_work["size_class"] = (_jnb_s.yb + 1).astype(float)

# Apply Jenks color classification (Decision 4)
scores  = df_work["avg_roberta_compound"]
_jnb_c  = mapclassify.NaturalBreaks(scores.values, k=N_COLOR_BUCKETS)
_breaks = _jnb_c.bins
_lo     = scores.min()
_labels = []
for _hi in _breaks:
    _labels.append(f"{_lo:.2f} to {_hi:.2f}")
    _lo = _hi
df_work["color_class"] = pd.cut(scores, bins=[-float("inf")] + list(_breaks), labels=_labels)
_palette   = pc.sample_colorscale(COLOR_SCALE, [i / (N_COLOR_BUCKETS - 1) for i in range(N_COLOR_BUCKETS)])
_color_map = {label: color for label, color in zip(_labels, _palette)}

base = {
    "lat": "latitude",
    "lon": "longitude",
    "size": "size_class",
    "color": "color_class",
    "hover_name": "place",
    "hover_data": {
        "avg_roberta_compound": ":.3f",
        "location_count": True,
        "size_class": False,
        "color_class": False,
        "latitude": False,
        "longitude": False,
    },
    "color_discrete_map": _color_map,
    "category_orders": {
        "color_class": _labels,
    },
    "size_max": 18,
    "map_style": MAP_STYLE,
    "height": 460,
}

for zoom_val, lat, lon, label in [
    (3,  38.0,  -96.0,  "National (zoom=3) — geoparser errors are now visible"),
    (6,  37.5,  -78.0,  "Virginia (zoom=6) — regional framing"),
    (11, 38.44, -78.87, "Harrisonburg (zoom=11) — campus-level detail"),
]:
    fig = px.scatter_map(
        df_work,
        center={
            "lat": lat,
            "lon": lon,
        },
        zoom=zoom_val,
        title=label, **base
    )
    fig.update_layout(
        margin={
            "r": 0,
            "t": 50,
            "l": 0,
            "b": 0,
        }
    )
    fig.show()

### ✍️ 7.1 Critical Activity - Creating a Custom Map

Below is a code cell with all the decisions you can make about your map. Dial in different variables to create your own custom map that tells your story.

In [ ]:
import mapclassify
import plotly.colors as pc

# ── Your design decisions — change each value and re-run ─────────────────────
MIN_COUNT       =   # ← Decision 1: minimum post count
PLACE_TYPES     =   # ← Decision 2: list of place types (e.g. ['City', 'Building']), or None for all
N_SIZE_CLASSES  =   # ← Decision 3b: number of Jenks size classes
N_COLOR_BUCKETS =   # ← Decision 4: number of Jenks color buckets
COLOR_SCALE     =   # ← Decision 5: Plotly color scale name (e.g. 'RdYlGn')
MAP_STYLE       =   # ← Decision 6: base map style (e.g. 'carto-positron')
CENTER          =   # ← Decision 7: map center, e.g. {"lat": 37.5, "lon": -78.0}
ZOOM            =   # ← Decision 7: zoom level

# ── Data preparation ──────────────────────────────────────────────────────────
if PLACE_TYPES is not None:
    df_work = df_reddit_place_sentiments[
        (df_reddit_place_sentiments["location_count"] >= MIN_COUNT) &
        df_reddit_place_sentiments["place_type"].isin(PLACE_TYPES)
    ].copy()
else:
    df_work = df_reddit_place_sentiments[
        df_reddit_place_sentiments["location_count"] >= MIN_COUNT
    ].copy()

_jnb_s = mapclassify.NaturalBreaks(df_work["location_count"].values, k=N_SIZE_CLASSES)
df_work["size_class"] = (_jnb_s.yb + 1).astype(float)

scores  = df_work["avg_roberta_compound"]
_jnb_c  = mapclassify.NaturalBreaks(scores.values, k=N_COLOR_BUCKETS)
_breaks = _jnb_c.bins
_lo     = scores.min()
_labels = []
for _hi in _breaks:
    _labels.append(f"{_lo:.2f} to {_hi:.2f}")
    _lo = _hi
df_work["color_class"] = pd.cut(scores, bins=[-float("inf")] + list(_breaks), labels=_labels)
_palette   = pc.sample_colorscale(COLOR_SCALE, [i / (N_COLOR_BUCKETS - 1) for i in range(N_COLOR_BUCKETS)])
_color_map = {label: color for label, color in zip(_labels, _palette)}

fig = px.scatter_map(
    df_work,
    lat="latitude", lon="longitude",
    size="size_class",
    color="color_class",
    hover_name="place",
    hover_data={
        "avg_roberta_compound": ":.3f",
        "location_count": True,
        "size_class": False,
        "color_class": False,
        "latitude": False,
        "longitude": False,
    },
    color_discrete_map=_color_map,
    category_orders={
        "color_class": _labels,
    },
    size_max=18,
    map_style=MAP_STYLE,
    center=CENTER,
    zoom=ZOOM,
    height=650,
    title="Your Final Map"
)
fig.update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
)
fig.show()

> 💡 **Reflection:** Why did you make the map the way you did? What were your key decisions in terms of the variables? What is the story you are trying to tell?


---

## Lesson Summary

Here is what you covered in this lesson — and in the unit as a whole:

### The Full Pipeline
| Step | Lesson | Tool | What it produced |
|------|--------|------|-----------------|
| Load & clean data | 3 | pandas | Sentence-level DataFrame |
| Extract place names | 4 | spaCy NER + transformer NER | Entity spans per sentence |
| Resolve to coordinates | 4 | Geoparser + GeoNames | Latitude/longitude per mention |
| Score sentiment | 5.1 | VADER | `vader_sentiment` per sentence |
| Score sentiment | 5.2 | RoBERTa | `roberta_compound` per sentence |
| Aggregate & map | 6 | Plotly | Emotional geography |

### Key Concepts
- **Named Entity Recognition (NER)** — identifying place names (and other entities) in unstructured text
- **Geoparsing** — disambiguating place names and resolving them to geographic coordinates
- **Sentiment analysis** — scoring text on a positive/negative scale; rule-based (VADER) vs. transformer-based (RoBERTa)
- **Aggregation** — collapsing row-level data to a summary by group (here: by place)
- **Limitations compound** — every imperfect step in a pipeline introduces noise that accumulates in the final output

### What the Map Does — and Doesn't — Show
This map visualizes *how Reddit users wrote about places*, filtered through several imperfect models. It is not a ground-truth measure of how happy or unhappy those places are. That distinction — between the signal in the data and the reality it is meant to represent — is at the heart of critical data literacy.

---

➡️ **Next:** [Project: Mapping Emotions](../project_mapping_emotions/README.md)
